# Elliptic Dataset — Exploratory Data Analysis

Covers: class balance overall and per time step, degree distribution, neighbor-label homophily (does illicit cluster locally?), and feature distributions split by class. These numbers justify (or don't) using a GNN in Phase 3 — write findings in the markdown cells below each plot after running.

In [ ]:
import sys
sys.path.append('..')

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from src.data import ILLICIT, LICIT, UNKNOWN, load_raw

sns.set_theme(style='whitegrid')
data = load_raw()
print(f"nodes={len(data.features)}  edges={len(data.edges)}  features={len(data.feature_cols)}")

## 1. Class balance — overall

In [ ]:
counts = data.labels.value_counts().rename({ILLICIT: 'illicit', LICIT: 'licit', UNKNOWN: 'unknown'})
display(counts)
display((counts / counts.sum() * 100).round(2).rename('pct'))

## 2. Class balance per time step

Checks whether every time step has enough illicit examples for a usable split (informs the exact Phase 1 train/val/test cut).

In [ ]:
df = data.features[['time_step']].copy()
df['label'] = data.labels.values
per_step = df.groupby(['time_step', 'label']).size().unstack(fill_value=0)
per_step = per_step.rename(columns={ILLICIT: 'illicit', LICIT: 'licit', UNKNOWN: 'unknown'})

fig, ax = plt.subplots(figsize=(12, 4))
per_step[['illicit', 'licit']].plot(ax=ax)
ax.axvline(34.5, color='gray', linestyle='--', label='train/val cut')
ax.axvline(39.5, color='black', linestyle='--', label='val/test cut')
ax.set_ylabel('# labeled nodes')
ax.legend()
plt.show()
per_step

**Findings:** _fill in after running — e.g. does the illicit rate spike at any particular time step (the dataset is known to have a spike around a dark-market takedown), and does the chosen 34/5/10 split leave enough illicit examples in val/test?_

## 3. Degree distribution

In [ ]:
deg = pd.concat([data.edges['src'], data.edges['dst']]).value_counts()
deg = deg.reindex(data.features.index, fill_value=0)

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(deg, bins=50, log=True)
ax.set_xlabel('degree')
ax.set_ylabel('count (log scale)')
ax.set_title('Node degree distribution')
plt.show()
print(deg.describe())

## 4. Neighbor-label homophily

For each labeled node, what fraction of its neighbors share its label? High homophily is the main justification for a GNN — if illicit nodes cluster together, message passing should help; if not, a GNN has little structural signal to exploit.

In [ ]:
id_to_idx = {tx_id: i for i, tx_id in enumerate(data.features.index)}
labels_arr = data.labels.to_numpy()
n = len(labels_arr)

neighbor_lists = [[] for _ in range(n)]
src_idx = data.edges['src'].map(id_to_idx).to_numpy()
dst_idx = data.edges['dst'].map(id_to_idx).to_numpy()
for s, d in zip(src_idx, dst_idx):
    neighbor_lists[s].append(d)
    neighbor_lists[d].append(s)

homophily = []
for i in range(n):
    if labels_arr[i] == UNKNOWN or not neighbor_lists[i]:
        continue
    neigh_labels = labels_arr[neighbor_lists[i]]
    known = neigh_labels[neigh_labels != UNKNOWN]
    if len(known) == 0:
        continue
    homophily.append({'label': labels_arr[i], 'frac_same_label': (known == labels_arr[i]).mean()})

homophily_df = pd.DataFrame(homophily)
homophily_df.groupby('label')['frac_same_label'].mean().rename({ILLICIT: 'illicit', LICIT: 'licit'})

**Findings:** _fill in after running — this number is the key input to whether a GNN should be expected to help in Phase 3, and (per the PRD's core hypothesis) whether that help survives the leakage-free protocol in Phase 4._

## 5. Feature distributions by class

A handful of the most separating features (proxied here by absolute mean difference between classes).

In [ ]:
labeled = data.labeled_mask()
X = data.features.loc[labeled, data.feature_cols]
y = data.labels.loc[labeled]

mean_diff = (X[y == ILLICIT].mean() - X[y == LICIT].mean()).abs().sort_values(ascending=False)
top_features = mean_diff.head(6).index.tolist()

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, feat in zip(axes.flat, top_features):
    sns.kdeplot(X.loc[y == LICIT, feat], ax=ax, label='licit')
    sns.kdeplot(X.loc[y == ILLICIT, feat], ax=ax, label='illicit')
    ax.set_title(feat)
    ax.legend()
plt.tight_layout()
plt.show()

**Findings:** _fill in after running — do these features look like the kind Phase 2's XGBoost feature-importance plot should also surface? Cross-check once Phase 2 runs._